In [2]:
import pandas as pd

ruta = r"C:\Users\jparrap\Sitemas_Excel\datos_sharepoint_powerquery.xlsx"

df = pd.read_excel(ruta)

print(df.head())


                                              Título     Estado    NumFactura  \
0                  jesus.parra@bolsamercantil.com.co   Aprobado      ECOD2134   
1                  jesus.parra@bolsamercantil.com.co   Aprobado  GYYCZRKP0008   
2                  jesus.parra@bolsamercantil.com.co   Aprobado  GYYCZRKP0011   
3  ibqtjb.fa.sender@workflow.email.us-ashburn-1.o...   Aprobado  JTHEPL7Z0003   
4  ibqtjb.fa.sender@workflow.email.us-ashburn-1.o...  Pendiente      ECOD2245   

  FechaCorreo FechaFacturacion FechaVencimiento  \
0  2026-05-15       2026-03-20       2026-03-25   
1  2026-05-19       2026-03-13       2026-03-13   
2  2026-05-21       2026-05-13       2026-05-13   
3  2026-06-09       2026-06-01       2026-06-01   
4  2026-06-12       2026-06-02       2026-05-21   

                                         Descripcion Vendedor  \
0  BOLSA DE 15 HORAS MENSUALES DE SOPORTE TECNICO...       CI   
1  ChatGPT Business Subscription (per seat) 141 $...   OpenAI   
2  ChatGPT 

In [2]:
import pandas as pd
import sqlite3

ruta = r"C:\Users\jparrap\Sitemas_Excel\datos_sharepoint_powerquery.xlsx"

# EXTRACT
df = pd.read_excel(ruta)

# TRANSFORM
columnas = [
    "Estado",
    "NumFactura",
    "FechaFacturacion",
    "FechaVencimiento",
    "Descripcion",
    "Vendedor",
    "Provedor",
    "ValorCOP_M1",
    "ValorCOP_M2",
    "ValorTotalFactura"
]

df = df[columnas]

# LOAD
conn = sqlite3.connect("facturas.db")
df.to_sql("facturas", conn, if_exists="replace", index=False)

conn.close()

In [3]:
conn = sqlite3.connect("facturas.db")

df_check = pd.read_sql("SELECT * FROM facturas LIMIT 5", conn)

print(df_check)

conn.close()

      Estado    NumFactura FechaFacturacion FechaVencimiento  \
0   Aprobado      ECOD2134       2026-03-20       2026-03-25   
1   Aprobado  GYYCZRKP0008       2026-03-13       2026-03-13   
2   Aprobado  GYYCZRKP0011       2026-05-13       2026-05-13   
3   Aprobado  JTHEPL7Z0003       2026-06-01       2026-06-01   
4  Pendiente      ECOD2245       2026-06-02       2026-05-21   

                                         Descripcion Vendedor  \
0  BOLSA DE 15 HORAS MENSUALES DE SOPORTE TECNICO...       CI   
1  ChatGPT Business Subscription (per seat) 141 $...   OpenAI   
2  ChatGPT Business Subscription (per seat) 144 $...   OpenAI   
3  Remaining time on 10 x Team plan - Standard af...       AL   
4  BOLSA DE 15 HORAS MENSUALES DE SOPORTE TECNICO...       CI   

          Provedor ValorCOP_M1 ValorCOP_M2 ValorTotalFactura  
0  CODE LABS S A S   1.890.000   1.890.000              None  
1       OPENAI LLC  16.036.495  16.036.495         $4,230.00  
2       OPENAI LLC  13.911.318  13.

In [1]:
import pandas as pd
import pyodbc


# =========================
# CONFIG
# =========================

RUTA_EXCEL = r"C:\Users\jparrap\Sistemas_Excel\datos_sharepoint_powerquery.xlsx"
DB_PATH = r"C:\Users\jparrap\Documents\dbfacturas.accdb"

TRM = 4000


# =========================
# LIMPIEZA NUMERICA PRO
# =========================

def limpiar_y_detectar(x):

    if pd.isna(x):
        return None, None

    moneda = "COP"

    if isinstance(x, str):
        x = x.strip()

        if "$" in x:
            moneda = "USD"

        x = x.replace("$", "").replace(" ", "")

        if "." in x and "," not in x:
            x = x.replace(".", "")

        elif "," in x and "." in x:
            x = x.replace(",", "")

        elif "," in x:
            x = x.replace(",", "")

    try:
        return float(x), moneda
    except:
        return None, None


# =========================
# FIX NaN → NULL
# =========================

def safe_value(x):
    if pd.isna(x):
        return None
    return float(x)


# =========================
# EXTRACT
# =========================

def extract():
    return pd.read_excel(RUTA_EXCEL)


# =========================
# TRANSFORM
# =========================

def transform(df):

    columnas = [
        "Estado",
        "NumFactura",
        "FechaFacturacion",
        "FechaVencimiento",
        "FechaM2",  # 👈 NUEVO CAMPO
        "Descripcion",
        "Vendedor",
        "Provedor",
        "ValorCOP_M1",
        "ValorCOP_M2",
        "ValorTotalFactura"
    ]

    df = df[columnas].copy()

    # -------------------------
    # FECHAS
    # -------------------------

    df["FechaFacturacion"] = pd.to_datetime(df["FechaFacturacion"], errors="coerce")
    df["FechaVencimiento"] = pd.to_datetime(df["FechaVencimiento"], errors="coerce")
    df["FechaM2"] = pd.to_datetime(df["FechaM2"], errors="coerce")  # 👈 NUEVO

    # -------------------------
    # NUMERICOS
    # -------------------------

    columnas_num = [
        "ValorCOP_M1",
        "ValorCOP_M2",
        "ValorTotalFactura"
    ]

    for col in columnas_num:

        resultados = df[col].apply(limpiar_y_detectar)

        df[col] = resultados.apply(lambda x: x[0])

        if col == "ValorTotalFactura":
            df["Moneda"] = resultados.apply(lambda x: x[1])

    # -------------------------
    # USD → COP
    # -------------------------

    df["ValorTotalCOP"] = df.apply(
        lambda row: row["ValorTotalFactura"] * TRM
        if row.get("Moneda") == "USD" and pd.notna(row["ValorTotalFactura"])
        else row["ValorTotalFactura"],
        axis=1
    )

    # -------------------------
    # LIMPIEZA FACTURA
    # -------------------------

    df["NumFactura"] = df["NumFactura"].astype(str).str.strip()

    # -------------------------
    # CALIDAD
    # -------------------------

    df["calidad"] = (
        df["FechaFacturacion"].notna().astype(int)
        +
        df["FechaVencimiento"].notna().astype(int)
    )

    df = df.sort_values("calidad", ascending=False)
    df = df.drop_duplicates(subset=["NumFactura"], keep="first")
    df = df.drop(columns=["calidad"])

    return df


# =========================
# LOAD UPSERT ACCESS
# =========================

def load(df):

    conn = pyodbc.connect(
        r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
        rf"DBQ={DB_PATH};"
    )

    cursor = conn.cursor()

    insertados = 0
    actualizados = 0

    for _, row in df.iterrows():

        factura = row["NumFactura"]

        if factura in ["", "nan", None]:
            continue

        fecha_fact = (
            row["FechaFacturacion"].to_pydatetime()
            if pd.notna(row["FechaFacturacion"])
            else None
        )

        fecha_venc = (
            row["FechaVencimiento"].to_pydatetime()
            if pd.notna(row["FechaVencimiento"])
            else None
        )

        fecha_m2 = (  # 👈 NUEVO
            row["FechaM2"].to_pydatetime()
            if pd.notna(row["FechaM2"])
            else None
        )

        print("DEBUG:",
              safe_value(row["ValorCOP_M1"]),
              safe_value(row["ValorCOP_M2"]),
              safe_value(row["ValorTotalCOP"]),
              fecha_m2
        )

        # EXISTENCIA
        cursor.execute(
            """
            SELECT NumFactura
            FROM tb_Facturas
            WHERE NumFactura = ?
            """,
            (factura,)
        )

        existente = cursor.fetchone()

        # -------------------------
        # INSERT
        # -------------------------

        if existente is None:

            cursor.execute(
                """
                INSERT INTO tb_Facturas
                (
                    NumFactura,
                    Estado,
                    FechaFacturacion,
                    FechaVencimiento,
                    FechaM2,
                    Descripcion,
                    Vendedor,
                    Provedor,
                    ValorCOP_M1,
                    ValorCOP_M2,
                    ValorTotalFactura
                )
                VALUES (?,?,?,?,?,?,?,?,?,?,?)
                """,
                (
                    factura,
                    str(row["Estado"]),
                    fecha_fact,
                    fecha_venc,
                    fecha_m2,
                    str(row["Descripcion"]),
                    str(row["Vendedor"]),
                    str(row["Provedor"]),
                    safe_value(row["ValorCOP_M1"]),
                    safe_value(row["ValorCOP_M2"]),
                    safe_value(row["ValorTotalCOP"])
                )
            )

            insertados += 1

        # -------------------------
        # UPDATE
        # -------------------------

        else:

            cursor.execute(
                """
                UPDATE tb_Facturas
                SET
                    Estado=?,
                    FechaFacturacion=?,
                    FechaVencimiento=?,
                    FechaM2=?,
                    Descripcion=?,
                    Vendedor=?,
                    Provedor=?,
                    ValorCOP_M1=?,
                    ValorCOP_M2=?,
                    ValorTotalFactura=?
                WHERE NumFactura=?
                """,
                (
                    str(row["Estado"]),
                    fecha_fact,
                    fecha_venc,
                    fecha_m2,
                    str(row["Descripcion"]),
                    str(row["Vendedor"]),
                    str(row["Provedor"]),
                    safe_value(row["ValorCOP_M1"]),
                    safe_value(row["ValorCOP_M2"]),
                    safe_value(row["ValorTotalCOP"]),
                    factura
                )
            )

            actualizados += 1

    conn.commit()
    conn.close()

    return insertados, actualizados


# =========================
# MAIN
# =========================

def main():

    df = extract()
    print("Excel cargado:", df.shape)

    df = transform(df)
    print("Después de transformación:", df.shape)

    insertados, actualizados = load(df)

    print("✅ Proceso terminado")
    print(f"🆕 Insertados: {insertados}")
    print(f"🔄 Actualizados: {actualizados}")
    print(f"📊 Total procesados: {len(df)}")


# =========================
# RUN
# =========================

main()

Excel cargado: (20, 43)
Después de transformación: (15, 13)


Error: ('HY000', "[HY000] [Microsoft][Controlador ODBC Microsoft Access] No se pudo encontrar el archivo '(desconocido)'. (-1811) (SQLDriverConnect); [HY000] [Microsoft][Controlador ODBC Microsoft Access] No se pudo encontrar el archivo '(desconocido)'. (-1811)")

In [21]:
import os

print(DB_PATH)
print(os.path.exists(DB_PATH))


C:\Users\jparrap\Documents\facturas.accdb
False


In [1]:
import pyodbc

DB_PATH = r"C:\ruta\tu_base.accdb"

def get_connection():
    conn = pyodbc.connect(
        rf"DRIVER={{Microsoft Access Driver (*.mdb, *.accdb)}};DBQ={DB_PATH};"
    )
    return conn

In [8]:
print(df.columns.tolist())


['Título', 'Estado', 'NumFactura', 'FechaCorreo', 'FechaFacturacion', 'FechaVencimiento', 'Descripcion', 'Vendedor', 'Provedor', 'Servicio', 'MedioPago', 'ValorCOP_M1', 'FechaM2', 'ValorUSD_M2', 'ValorCOP_M2', 'MetodoPago', 'Proyecto', 'Aprobador', 'Observaciones', 'Asunto', 'MennsageId', 'ValorTotalFactura', 'Etiqueta de color', 'Id. de activo de cumplimiento normativo', 'ID', 'Tipo de contenido', 'Modificado', 'Creado', 'Creado por', 'Modificado por', 'Versión', 'Datos adjuntos', 'Editar', 'Tipo', 'Número secundario de elemento', 'Recuento secundario de carpetas', 'Configuración de la etiqueta', 'Etiqueta de retención', 'Etiqueta de retención aplicada', 'Usuario que ha aplicado la etiqueta:', 'El elemento es un registro', 'Aplicación creada por', 'Aplicación modificada por']


In [35]:
import pyodbc

DB_PATH = r"C:\Users\jparrap\Sitemas_Excel\FacturasPRY\database\dbfacturas.accdb"

conn = pyodbc.connect(
    rf"DRIVER={{Microsoft Access Driver (*.mdb, *.accdb)}};DBQ={DB_PATH};"
)

cursor = conn.cursor()

cursor.execute("SELECT TOP 1 * FROM tb_Facturas")

print([col[0] for col in cursor.description])

conn.close()

['Id', 'NumFactura', 'Estado', 'FechaFacturacion', 'FechaVencimiento', 'Descripcion', 'Vendedor', 'Provedor', 'ValorCOP_M1', 'ValorCOP_M2', 'ValorTotalFactura', 'FechaM2', 'IdArea', 'IdVertical']


In [2]:
import pandas as pd
import pyodbc

# =========================
# CONFIG
# =========================

RUTA_EXCEL = r"C:\Users\jparrap\Documents\Enlace_Sharepoint.xlsx"
DB_PATH = r"C:\Users\jparrap\Sistemas_Excel\FacturasPRY\database\dbfacturas.accdb"

TRM = 4000


# =========================
# LIMPIEZA NUMERICA
# =========================

def limpiar_y_detectar(x):

    if pd.isna(x):
        return None, None

    moneda = "COP"

    if isinstance(x, str):
        x = x.strip()

        if "$" in x:
            moneda = "USD"

        x = x.replace("$", "").replace(" ", "")

        if "." in x and "," not in x:
            x = x.replace(".", "")
        elif "," in x and "." in x:
            x = x.replace(",", "")
        elif "," in x:
            x = x.replace(",", "")

    try:
        return float(x), moneda

    except:
        return None, None


def safe_value(x):

    return None if pd.isna(x) else float(x)


def safe_str(x):

    return None if pd.isna(x) else str(x)


# =========================
# EXTRACT
# =========================

def extract():

    df = pd.read_excel(RUTA_EXCEL)

    # Limpiar nombres de columnas
    df.columns = df.columns.str.strip()

    return df


# =========================
# TRANSFORM
# =========================

def transform(df):

    # =========================
    # COLUMNAS DEL EXCEL
    # =========================

    columnas = [
        "Estado",
        "NumFactura",
        "FechaCorreo",
        "FechaVencimiento",
        "FechaM2",
        "Descripcion",
        "Vendedor",
        "Provedor",
        "Area",
        "Agrupaciones",
        "Tipo_Gasto",
        "ValorCOP_M1",
        "ValorCOP_M2",
        "ValorTotalFactura"
    ]

    df = df[columnas].copy()

    # =========================
    # FECHAS
    # =========================

    # FechaCorreo continúa siendo
    # la fuente de verdad para FechaFacturacion.

    df = df.rename(columns={
        "FechaCorreo": "FechaFacturacion"
    })

    df["FechaFacturacion"] = pd.to_datetime(
        df["FechaFacturacion"],
        errors="coerce"
    )

    df["FechaVencimiento"] = pd.to_datetime(
        df["FechaVencimiento"],
        errors="coerce"
    )

    df["FechaM2"] = pd.to_datetime(
        df["FechaM2"],
        errors="coerce"
    )

    # =========================
    # AREA
    # =========================

    df["Area"] = (
        df["Area"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # =========================
    # AGRUPACIONES
    # =========================

    # Se conserva como texto.
    # Posteriormente se buscará su Id
    # en tb_Agrupacion.

    df["Agrupaciones"] = (
        df["Agrupaciones"]
        .apply(
            lambda x:
                None
                if pd.isna(x)
                else str(x).strip()
        )
    )

    # =========================
    # TIPO DE GASTO
    # =========================

    # La fuente en Excel se llama:
    # Tipo_Gasto
    #
    # En la base de datos el campo se llama:
    # Tipo
    #
    # Valores permitidos:
    # OPEX / CAPEX

    df["Tipo"] = (
        df["Tipo_Gasto"]
        .apply(
            lambda x:
                None
                if pd.isna(x)
                else str(x).strip().upper()
        )
    )

    # Solo permitir OPEX y CAPEX.
    # Cualquier otro valor queda NULL.

    df["Tipo"] = df["Tipo"].where(
        df["Tipo"].isin(["OPEX", "CAPEX"]),
        None
    )

    # =========================
    # VALORES NUMERICOS
    # =========================

    for col in [
        "ValorCOP_M1",
        "ValorCOP_M2",
        "ValorTotalFactura"
    ]:

        res = df[col].apply(
            limpiar_y_detectar
        )

        df[col] = res.apply(
            lambda x: x[0]
        )

        if col == "ValorTotalFactura":

            df["Moneda"] = res.apply(
                lambda x: x[1]
            )

    # =========================
    # VALOR TOTAL COP
    # =========================

    df["ValorTotalCOP"] = df.apply(
        lambda r:
            r["ValorTotalFactura"] * TRM
            if r["Moneda"] == "USD"
            and pd.notna(
                r["ValorTotalFactura"]
            )
            else r["ValorTotalFactura"],
        axis=1
    )

    # =========================
    # NUMERO FACTURA
    # =========================

    df["NumFactura"] = (
        df["NumFactura"]
        .astype(str)
        .str.strip()
    )

    # =========================
    # AÑO
    # =========================

    df["Ano"] = (
        df["FechaFacturacion"]
        .dt.year
    )

    # =========================
    # CALIDAD
    # =========================

    df["calidad"] = (
        df["FechaFacturacion"]
        .notna()
        .astype(int)
        +
        df["FechaVencimiento"]
        .notna()
        .astype(int)
    )

    df = df.sort_values(
        "calidad",
        ascending=False
    )

    df = df.drop_duplicates(
        subset=["NumFactura"]
    )

    df = df.drop(
        columns=["calidad"]
    )

    return df


# =========================
# AREA
# =========================

def get_area_id(cursor, nombre):

    if nombre in ["", "nan", None]:

        raise ValueError(
            "Factura sin área"
        )

    cursor.execute(
        """
        SELECT Id
        FROM tb_Area
        WHERE UCASE(Nombre)=?
        """,
        (nombre.upper(),)
    )

    r = cursor.fetchone()

    if r:
        return r[0]

    raise ValueError(
        f"Área no existe: {nombre}"
    )


# =========================
# PROVEEDOR
# =========================

def get_proveedor_id(
    cursor,
    nombre,
    faltantes_dict,
    factura
):

    if (
        nombre in ["", "nan", None]
        or pd.isna(nombre)
    ):

        clave = "(SIN PROVEEDOR)"

        faltantes_dict.setdefault(
            clave,
            []
        ).append(factura)

        return None

    nombre = str(nombre).strip()

    cursor.execute(
        """
        SELECT Id
        FROM tb_Proveedor
        WHERE UCASE(Nombre)=?
        """,
        (nombre.upper(),)
    )

    r = cursor.fetchone()

    if r:
        return r[0]

    faltantes_dict.setdefault(
        nombre,
        []
    ).append(factura)

    return None


# =========================
# AGRUPACION
# =========================

def get_agrupacion_id(cursor, nombre):

    # Si viene vacío, se asigna
    # SIN ASIGNAR = Id 1.

    if (
        nombre in ["", "nan", None]
        or pd.isna(nombre)
    ):

        return 1

    nombre = str(nombre).strip()

    cursor.execute(
        """
        SELECT Id
        FROM tb_Agrupacion
        WHERE UCASE(Nombre)=?
        """,
        (nombre.upper(),)
    )

    r = cursor.fetchone()

    if r:
        return r[0]

    # No detenemos el ETL si la agrupación
    # todavía no existe.

    print(
        f"⚠️ Agrupación no existe: "
        f"{nombre}. "
        f"Se asignará IdAgrupacion=1"
    )

    return 1


# =========================
# LOAD
# =========================

def load(df):

    conn = pyodbc.connect(
        r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
        rf"DBQ={DB_PATH};"
    )

    cursor = conn.cursor()

    insertados = 0
    actualizados = 0
    rechazados = 0
    faltantes = {}

    for _, row in df.iterrows():

        factura = row["NumFactura"]

        if factura in ["", "nan", None]:
            continue

        # =========================
        # AREA
        # =========================

        try:

            id_area = get_area_id(
                cursor,
                row["Area"]
            )

        except Exception as e:

            print(
                f"⚠️ {factura} rechazada:",
                e
            )

            rechazados += 1

            continue

        # =========================
        # PROVEEDOR
        # =========================

        id_proveedor = get_proveedor_id(
            cursor,
            row["Provedor"],
            faltantes,
            factura
        )

        # =========================
        # AGRUPACION
        # =========================

        id_agrupacion = get_agrupacion_id(
            cursor,
            row["Agrupaciones"]
        )

        # =========================
        # FECHAS
        # =========================

        fecha_fact = (
            row["FechaFacturacion"].to_pydatetime()
            if pd.notna(
                row["FechaFacturacion"]
            )
            else None
        )

        fecha_venc = (
            row["FechaVencimiento"].to_pydatetime()
            if pd.notna(
                row["FechaVencimiento"]
            )
            else None
        )

        fecha_m2 = (
            row["FechaM2"].to_pydatetime()
            if pd.notna(
                row["FechaM2"]
            )
            else None
        )

        # =========================
        # MES
        # =========================

        mes = (
            fecha_m2.month
            if fecha_m2
            else None
        )

        # =========================
        # MES PENDIENTE
        # =========================

        mes_pendiente = (
            fecha_fact.month
            if fecha_fact
            else None
        )

        # =========================
        # VERIFICAR FACTURA
        # =========================

        cursor.execute(
            """
            SELECT NumFactura
            FROM tb_Facturas
            WHERE NumFactura=?
            """,
            (factura,)
        )

        existe = cursor.fetchone()

        # =========================
        # INSERT
        # =========================

        if existe is None:

            cursor.execute(
                """
                INSERT INTO tb_Facturas
                (
                    NumFactura,
                    Estado,
                    Tipo,
                    FechaFacturacion,
                    FechaVencimiento,
                    FechaM2,
                    Descripcion,
                    Vendedor,
                    IdProveedor,
                    IdArea,
                    IdAgrupacion,
                    ValorCOP_M1,
                    ValorCOP_M2,
                    ValorTotalFactura,
                    Mes,
                    Ano,
                    Mes_pendiente
                )
                VALUES (
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?,
                    ?
                )
                """,
                (
                    factura,

                    safe_str(
                        row["Estado"]
                    ),

                    safe_str(
                        row["Tipo"]
                    ),

                    fecha_fact,

                    fecha_venc,

                    fecha_m2,

                    safe_str(
                        row["Descripcion"]
                    ),

                    safe_str(
                        row["Vendedor"]
                    ),

                    id_proveedor,

                    id_area,

                    id_agrupacion,

                    safe_value(
                        row["ValorCOP_M1"]
                    ),

                    safe_value(
                        row["ValorCOP_M2"]
                    ),

                    safe_value(
                        row["ValorTotalCOP"]
                    ),

                    mes,

                    int(row["Ano"])
                    if pd.notna(row["Ano"])
                    else None,

                    mes_pendiente
                )
            )

            insertados += 1

        # =========================
        # UPDATE
        # =========================

        else:

            cursor.execute(
                """
                UPDATE tb_Facturas SET
                    Estado=?,
                    Tipo=?,
                    FechaFacturacion=?,
                    FechaVencimiento=?,
                    FechaM2=?,
                    Descripcion=?,
                    Vendedor=?,
                    IdProveedor=?,
                    IdArea=?,
                    IdAgrupacion=?,
                    ValorCOP_M1=?,
                    ValorCOP_M2=?,
                    ValorTotalFactura=?,
                    Mes=?,
                    Ano=?,
                    Mes_pendiente=?
                WHERE NumFactura=?
                """,
                (
                    safe_str(
                        row["Estado"]
                    ),

                    safe_str(
                        row["Tipo"]
                    ),

                    fecha_fact,

                    fecha_venc,

                    fecha_m2,

                    safe_str(
                        row["Descripcion"]
                    ),

                    safe_str(
                        row["Vendedor"]
                    ),

                    id_proveedor,

                    id_area,

                    id_agrupacion,

                    safe_value(
                        row["ValorCOP_M1"]
                    ),

                    safe_value(
                        row["ValorCOP_M2"]
                    ),

                    safe_value(
                        row["ValorTotalCOP"]
                    ),

                    mes,

                    int(row["Ano"])
                    if pd.notna(row["Ano"])
                    else None,

                    mes_pendiente,

                    factura
                )
            )

            actualizados += 1

    # =========================
    # COMMIT
    # =========================

    conn.commit()

    conn.close()

    return (
        insertados,
        actualizados,
        rechazados,
        faltantes
    )


# =========================
# MOSTRAR FALTANTES
# =========================

def mostrar_faltantes(faltantes):

    if not faltantes:

        print(
            "\n✅ Sin proveedores faltantes"
        )

        return

    print(
        "\n⚠️ PROVEEDORES FALTANTES:"
    )

    print("=" * 50)

    for prov, facts in faltantes.items():

        print(
            f"{prov} -> {', '.join(facts)}"
        )


# =========================
# MAIN
# =========================

def main():

    df = extract()

    df = transform(df)

    # =========================
    # VALIDACION TIPO
    # =========================

    print("\n==============================")
    print("TIPOS DE GASTO DETECTADOS")
    print("==============================")

    print(
        df["Tipo"]
        .value_counts(dropna=False)
    )

    ins, upd, rej, falt = load(df)

    print("\n✅ ETL COMPLETO")

    print(
        "Insertados:",
        ins
    )

    print(
        "Actualizados:",
        upd
    )

    print(
        "Rechazados:",
        rej
    )

    mostrar_faltantes(falt)


main()


TIPOS DE GASTO DETECTADOS
Tipo
OPEX     54
CAPEX     8
Name: count, dtype: int64

✅ ETL COMPLETO
Insertados: 0
Actualizados: 62
Rechazados: 0

✅ Sin proveedores faltantes


In [4]:
df = pd.read_excel(RUTA_EXCEL)

print("COLUMNAS REALES DEL EXCEL:")
print("=" * 60)

for i, col in enumerate(df.columns):
    print(i, repr(col))

COLUMNAS REALES DEL EXCEL:
0 'Título'
1 'Estado'
2 'NumFactura'
3 'FechaCorreo'
4 'FechaFacturacion'
5 'FechaVencimiento'
6 'FechaM2'
7 'Descripcion'
8 'Vendedor'
9 'Provedor'
10 'Servicio'
11 'MedioPago'
12 'ValorCOP_M1'
13 'ValorUSD_M2'
14 'ValorCOP_M2'
15 'MetodoPago'
16 'Proyecto'
17 'Aprobador'
18 'Observaciones'
19 'Asunto'
20 'MennsageId'
21 'ValorTotalFactura'
22 'Area'
23 'Tipo_Gasto'
24 'Etiqueta de color'
25 'Id. de activo de cumplimiento normativo'
26 'ID'
27 'Tipo de contenido'
28 'Modificado'
29 'Creado'
30 'Creado por'
31 'Modificado por'
32 'Versión'
33 'Datos adjuntos'
34 'Editar'
35 'Tipo'
36 'Número secundario de elemento'
37 'Recuento secundario de carpetas'
38 'Configuración de la etiqueta'
39 'Etiqueta de retención'
40 'Etiqueta de retención aplicada'
41 'Usuario que ha aplicado la etiqueta:'
42 'El elemento es un registro'
43 'Aplicación creada por'
44 'Aplicación modificada por'
